# ORB-BSCNN: preliminary PTB-XL ECG experiment

**Dataset:** PTB-XL (PhysioNet), 100 Hz version.

Nodes of the complex are the 12 ECG leads (I, II, III, aVR, aVL, aVF, V1-V6). Edges connect
leads whose RMS signal is correlated above a threshold on the training set, and triangles
are 3-cliques of mutually correlated leads.

**Pipeline**
1. Download PTB-XL metadata and a stratified subset of 100 Hz records (`N_RECORDS`, default 2000).
2. Label each record as NORM (0) or abnormal (1) from the diagnostic superclass.
3. Represent each lead by five summary statistics (mean, std, min, max, RMS), giving a `[12 leads x 5 stats]` signal per record.
4. Build the lead-correlation complex from the training set.
5. Train the residual binarized simplicial convolution model on this signal.
6. Compare with MLP and Random Forest baselines.

**Limitations:** single 80/20 split at the record level (records from the same patient may fall
in both partitions), a 2,000-record subsample, and hand-crafted features rather than raw waveforms.
The results are preliminary.

## 1. Setup

In [ ]:
!pip -q install wfdb torch torchvision scikit-learn networkx pandas numpy matplotlib --upgrade


In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F
import wfdb
import os, ast, ssl, urllib.request, time, ast
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              confusion_matrix, roc_curve)
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

DATA_DIR = "/content/ptbxl"
os.makedirs(DATA_DIR, exist_ok=True)
N_RECORDS = 2000  # size of the stratified subsample (PTB-XL has 21,837 records)

## 2. Download PTB-XL metadata

`ptbxl_database.csv` (per-record metadata + diagnostic scp_codes) and `scp_statements.csv`
(maps scp_codes to diagnostic superclasses: NORM, MI, STTC, CD, HYP) - both small files.


In [ ]:
PHYSIONET_BASE = "https://physionet.org/files/ptb-xl/1.0.3/"

def download_file(url, dest):
    if not os.path.exists(dest):
        urllib.request.urlretrieve(url, dest)
        print(f"Downloaded {dest}")
    else:
        print(f"Already have {dest}")

download_file(PHYSIONET_BASE + "ptbxl_database.csv", f"{DATA_DIR}/ptbxl_database.csv")
download_file(PHYSIONET_BASE + "scp_statements.csv", f"{DATA_DIR}/scp_statements.csv")

meta = pd.read_csv(f"{DATA_DIR}/ptbxl_database.csv", index_col="ecg_id")
meta.scp_codes = meta.scp_codes.apply(ast.literal_eval)

scp_df = pd.read_csv(f"{DATA_DIR}/scp_statements.csv", index_col=0)
scp_df = scp_df[scp_df.diagnostic == 1]

print("Metadata rows:", meta.shape[0])
meta.head()


In [ ]:
def aggregate_diagnostic(scp_codes):
    classes = set()
    for code in scp_codes.keys():
        if code in scp_df.index:
            classes.add(scp_df.loc[code].diagnostic_class)
    return list(classes)

meta["diagnostic_superclass"] = meta.scp_codes.apply(aggregate_diagnostic)

def label_binary(classes):
    if len(classes) == 0:
        return None  # unlabeled / uncertain, drop
    if classes == ["NORM"]:
        return 0  # normal
    return 1  # any abnormal diagnostic class present

meta["target"] = meta["diagnostic_superclass"].apply(label_binary)
meta_labeled = meta.dropna(subset=["target"]).copy()
meta_labeled["target"] = meta_labeled["target"].astype(int)

print("Labeled records:", meta_labeled.shape[0])
print(meta_labeled["target"].value_counts())

# Subsample to N_RECORDS for reasonable Colab runtime (stratified)
if len(meta_labeled) > N_RECORDS:
    meta_sub, _ = train_test_split(meta_labeled, train_size=N_RECORDS,
                                    stratify=meta_labeled["target"], random_state=SEED)
else:
    meta_sub = meta_labeled

print("Using subset of", meta_sub.shape[0], "records")


## 3. Download waveforms and extract per-lead statistical signals

Each 100 Hz record is a `[1000 samples x 12 leads]` array covering 10 seconds. Five summary
statistics per lead (mean, std, min, max, RMS) are used as the node signal.

In [ ]:
LEAD_NAMES = ["I","II","III","aVR","aVL","aVF","V1","V2","V3","V4","V5","V6"]

def download_and_extract(ecg_id, filename_lr):
    # filename_lr is like "records100/00000/00001_lr" (no extension)
    local_path = f"{DATA_DIR}/{filename_lr}"
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    for ext in [".hea", ".dat"]:
        remote_url = PHYSIONET_BASE + filename_lr + ext
        local_file = local_path + ext
        if not os.path.exists(local_file):
            try:
                urllib.request.urlretrieve(remote_url, local_file)
            except Exception as e:
                return None
    try:
        record = wfdb.rdrecord(local_path)
        sig = record.p_signal  # [1000, 12]
    except Exception:
        return None

    stats = []
    for lead_idx in range(12):
        lead_sig = sig[:, lead_idx]
        stats.append([lead_sig.mean(), lead_sig.std(), lead_sig.min(),
                       lead_sig.max(), np.sqrt(np.mean(lead_sig ** 2))])
    return np.array(stats, dtype=np.float32)  # [12 leads, 5 stats]


features_list = []
targets_list = []
kept_ids = []

t0 = time.time()
for i, (ecg_id, row) in enumerate(meta_sub.iterrows()):
    feat = download_and_extract(ecg_id, row["filename_lr"])
    if feat is not None:
        features_list.append(feat)
        targets_list.append(row["target"])
        kept_ids.append(ecg_id)
    if (i + 1) % 200 == 0:
        elapsed = time.time() - t0
        print(f"{i+1}/{len(meta_sub)} processed, {len(features_list)} kept, {elapsed:.0f}s elapsed")

X_leads = np.stack(features_list)   # [n_samples, 12, 5]
y_ptbxl = np.array(targets_list, dtype=np.int64)
print("Final shape:", X_leads.shape, " labels:", np.bincount(y_ptbxl))


## 4. Train/test split and per-lead standardization

Each of the 5 statistics is standardized independently across the training set (fit on
train only, to avoid leakage), keeping the `[n, 12, 5]` shape.


In [ ]:
idx_train, idx_test = train_test_split(
    np.arange(len(y_ptbxl)), test_size=0.2, stratify=y_ptbxl, random_state=SEED)

Xtr_raw, Xte_raw = X_leads[idx_train], X_leads[idx_test]
ytr, yte = y_ptbxl[idx_train], y_ptbxl[idx_test]

# Standardize each of the 5 stat-channels independently, fit on train only
n_stats = Xtr_raw.shape[2]
means = Xtr_raw.reshape(-1, n_stats).mean(axis=0)
stds = Xtr_raw.reshape(-1, n_stats).std(axis=0) + 1e-8

Xtr = (Xtr_raw - means) / stds
Xte = (Xte_raw - means) / stds

print("Train:", Xtr.shape, "Test:", Xte.shape)


## 5. Lead-correlation simplicial complex

Nodes = 12 ECG leads. Edges connect leads whose RMS signal is correlated above a threshold
across training records. Triangles are 3-cliques of mutually correlated leads.

In [ ]:
def build_lead_complex(Xtr_stats, stat_idx=4, corr_threshold=0.3):
    # Xtr_stats: [n, 12, n_stats]; use one stat channel (default RMS, idx=4) to build the graph
    lead_signal = Xtr_stats[:, :, stat_idx]  # [n, 12]
    corr = np.corrcoef(lead_signal, rowvar=False)  # [12, 12]
    n_leads = corr.shape[0]

    G = nx.Graph()
    G.add_nodes_from(range(n_leads))
    for i in range(n_leads):
        for j in range(i + 1, n_leads):
            if abs(corr[i, j]) >= corr_threshold:
                G.add_edge(i, j)

    edges = list(G.edges())
    edge_index = {tuple(sorted(e)): idx for idx, e in enumerate(edges)}
    triangles = sorted(set(tuple(sorted(c)) for c in nx.enumerate_all_cliques(G) if len(c) == 3))
    n_edges, n_tri = len(edges), len(triangles)

    B1 = np.zeros((n_leads, n_edges), dtype=np.float32)
    for j, (u, v) in enumerate(edges):
        B1[u, j] = -1.0; B1[v, j] = 1.0

    B2 = np.zeros((n_edges, n_tri), dtype=np.float32)
    for t_idx, (a, b, c) in enumerate(triangles):
        for (u, v), sign in [((a, b), 1.0), ((b, c), 1.0), ((a, c), -1.0)]:
            key = tuple(sorted((u, v)))
            if key in edge_index:
                B2[edge_index[key], t_idx] = sign

    print(f"Lead graph: {n_leads} nodes, {n_edges} edges, {n_tri} triangles (threshold={corr_threshold})")
    print("Edges:", [(LEAD_NAMES[u], LEAD_NAMES[v]) for u, v in edges])
    return {"n_nodes": n_leads, "B1": B1, "B2": B2}


def hodge_laplacian_L0(cplx):
    B1 = cplx["B1"]
    L0 = B1 @ B1.T
    return L0


def normalize_laplacian(L):
    d = np.diag(L).copy()
    d[d == 0] = 1.0
    D_inv_sqrt = np.diag(1.0 / np.sqrt(d))
    return D_inv_sqrt @ L @ D_inv_sqrt

lead_cplx = build_lead_complex(Xtr, stat_idx=4, corr_threshold=0.3)
L0_leads = hodge_laplacian_L0(lead_cplx)
L0_leads_norm = torch.tensor(normalize_laplacian(L0_leads), dtype=torch.float32, device=device)


## 6. Binarized Simplicial Convolution (multi-dim node input version)

Same STE binarization + residual design as the UCI notebooks, generalized to accept a
`[batch, n_nodes, n_stats]` input (5 stats per lead) instead of a scalar per node.


In [ ]:
class BinarizeSTE(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return torch.sign(x + 1e-12)

    @staticmethod
    def backward(ctx, grad_output):
        (x,) = ctx.saved_tensors
        grad = grad_output.clone()
        grad[x.abs() > 1] = 0
        return grad

binarize = BinarizeSTE.apply


class BatchedBiSimplicialConv(nn.Module):
    def __init__(self, in_dim, out_dim, laplacian, k_hops=2):
        super().__init__()
        self.register_buffer("L", laplacian)
        self.k_hops = k_hops
        self.theta = nn.Parameter(torch.randn(k_hops + 1, in_dim, out_dim) * 0.1)
        self.bn = nn.BatchNorm1d(out_dim)

    def forward(self, X):
        out = 0
        Lx = X
        for h in range(self.k_hops + 1):
            out = out + Lx @ self.theta[h]
            Lx = torch.einsum("ij,bjf->bif", self.L, Lx)
        b, n, f = out.shape
        out = self.bn(out.reshape(b * n, f)).reshape(b, n, f)
        return out, binarize(out)


class ResidualBatchedBlock(nn.Module):
    def __init__(self, dim, laplacian, k_hops=2):
        super().__init__()
        self.conv = BatchedBiSimplicialConv(dim, dim, laplacian, k_hops)
        self.act = nn.PReLU()

    def forward(self, X):
        pre_bin, post_bin = self.conv(X)
        return self.act(post_bin + pre_bin) + X


class ECGLeadORB_BSCNN(nn.Module):
    """Lead-graph model: each patient is a signal over the fixed 12-lead complex."""
    def __init__(self, in_dim, hidden_dim, laplacian, n_blocks=2, k_hops=2, n_classes=2):
        super().__init__()
        self.input_proj = nn.Linear(in_dim, hidden_dim)  # in_dim = n_stats per lead (5)
        self.blocks = nn.ModuleList([ResidualBatchedBlock(hidden_dim, laplacian, k_hops) for _ in range(n_blocks)])
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2), nn.PReLU(), nn.Dropout(0.3),
            nn.Linear(hidden_dim // 2, n_classes))

    def forward(self, X):
        # X: [batch, 12, in_dim]
        h = self.input_proj(X)          # [batch, 12, hidden_dim]
        for block in self.blocks:
            h = block(h)
        pooled = h.mean(dim=1)          # mean-pool over leads
        return self.classifier(pooled)


## 7. Train ECGLeadORB_BSCNN

In [ ]:
Xtr_t = torch.tensor(Xtr, dtype=torch.float32, device=device)
ytr_t = torch.tensor(ytr, dtype=torch.long, device=device)
Xte_t = torch.tensor(Xte, dtype=torch.float32, device=device)

criterion = nn.CrossEntropyLoss()

def full_metrics(y_true, preds, probs):
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
    return {"Accuracy": accuracy_score(y_true, preds), "Sensitivity": tp / (tp + fn),
            "Specificity": tn / (tn + fp), "F1": f1_score(y_true, preds),
            "AUC": roc_auc_score(y_true, probs)}

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

model_ecg = ECGLeadORB_BSCNN(in_dim=n_stats, hidden_dim=32, laplacian=L0_leads_norm,
                              n_blocks=2, k_hops=2).to(device)
opt = torch.optim.AdamW(model_ecg.parameters(), lr=1e-2, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=15)

EPOCHS = 200
model_ecg.train()
for epoch in range(EPOCHS):
    opt.zero_grad()
    logits = model_ecg(Xtr_t)
    loss = criterion(logits, ytr_t)
    loss.backward(); opt.step(); sched.step(loss)
    if (epoch + 1) % 20 == 0:
        with torch.no_grad():
            acc = (logits.argmax(dim=1) == ytr_t).float().mean().item()
        print(f"Epoch {epoch+1:3d} | loss {loss.item():.4f} | train acc {acc:.4f}")

model_ecg.eval()
with torch.no_grad():
    logits_te = model_ecg(Xte_t)
    probs_te = F.softmax(logits_te, dim=1)[:, 1].cpu().numpy()
    preds_te = logits_te.argmax(dim=1).cpu().numpy()

metrics_ecg = full_metrics(yte, preds_te, probs_te)
print("ECGLeadORB-BSCNN:", metrics_ecg)


## 8. Baselines: MLP (flattened) and Random Forest

In [ ]:
Xtr_flat = Xtr.reshape(Xtr.shape[0], -1)  # [n, 60]
Xte_flat = Xte.reshape(Xte.shape[0], -1)

class MLP(nn.Module):
    def __init__(self, in_dim, hidden=32, n_classes=2):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(),
                                  nn.Linear(hidden, hidden), nn.ReLU(),
                                  nn.Linear(hidden, n_classes))
    def forward(self, x): return self.net(x)

mlp = MLP(Xtr_flat.shape[1]).to(device)
opt_mlp = torch.optim.Adam(mlp.parameters(), lr=1e-3)
Xtr_flat_t = torch.tensor(Xtr_flat, dtype=torch.float32, device=device)
Xte_flat_t = torch.tensor(Xte_flat, dtype=torch.float32, device=device)

mlp.train()
for _ in range(200):
    opt_mlp.zero_grad()
    loss = criterion(mlp(Xtr_flat_t), ytr_t)
    loss.backward(); opt_mlp.step()

mlp.eval()
with torch.no_grad():
    mlp_probs = F.softmax(mlp(Xte_flat_t), dim=1)[:, 1].cpu().numpy()
    mlp_preds = mlp(Xte_flat_t).argmax(dim=1).cpu().numpy()
metrics_mlp = full_metrics(yte, mlp_preds, mlp_probs)

rf = RandomForestClassifier(n_estimators=300, random_state=SEED)
rf.fit(Xtr_flat, ytr)
rf_preds = rf.predict(Xte_flat)
rf_probs = rf.predict_proba(Xte_flat)[:, 1]
metrics_rf = full_metrics(yte, rf_preds, rf_probs)

pd.DataFrame({
    "ECGLeadORB-BSCNN (proposed)": metrics_ecg,
    "MLP (flattened)": metrics_mlp,
    "Random Forest": metrics_rf,
}).T
